# Is That a Peak? Detecting Real Bands and Rejecting Fake Ones
### 3.4 — Peak Detection and Picking

Finding peaks sounds like the easy part. Your eye does it instantly: there's a
band at 450 nm, another near 690. But the moment you ask a computer to do it,
the question gets slippery. A peak is a *local maximum* — a point higher than its
neighbours — and a noisy spectrum has **hundreds** of local maxima, almost all of
them noise. So "find the local maxima" is not a peak detector. It's a noise
detector with delusions of grandeur.

This notebook is **not** "how to call `scipy.signal.find_peaks()`." That function
takes one line. The hard part — the part that decides whether your results are
real — is everything around it:

- what actually *makes* something a peak, beyond "it's a bump";
- the difference between **height** (how tall) and **prominence** (how much it
  stands out) — and why prominence is the idea that matters most;
- separating **signal from noise**, and the unavoidable trade-off between
  **false positives** (peaks that aren't there) and **false negatives** (real
  peaks you missed);
- **overlapping bands**, where two peaks hide inside one bump;
- choosing the four parameters — **height, prominence, distance, width** —
  deliberately instead of by trial and error;
- and how **smoothing** (3.2) and **baseline correction** (3.3) change peak
  detection more than the detector's own settings do.

Because we build every spectrum with `uvvis.simulate()`, we always **know the
true peak positions**. That lets us *grade* each attempt — counting exactly how
many real peaks we found and how many we invented — instead of trusting that the
picture looks convincing.

> **Where this sits:** this lesson follows **3.2 (Savitzky–Golay Smoothing)** and
> **3.3 (AsLS Baseline Correction)**. Those notebooks clean the spectrum; this one
> reads peaks off it. We'll show directly why that order matters.

**What we'll cover**

1. Setup
2. What even is a peak? (and why "local maximum" isn't enough)
3. A realistic test spectrum — with known truth
4. The naive detector, and why it fails
5. Height — the first, and most fragile, filter
6. Prominence — how much a peak stands out
7. Distance — minimum separation between peaks
8. Width — rejecting spikes, selecting bands
9. Putting the four knobs together
10. Grading against ground truth: false positives vs. false negatives
11. Preprocessing changes everything: smoothing and baseline correction
12. The overlapping-peak limit
13. A reusable `detect_peaks()` helper

## 1. Setup

In [ ]:
# Standard scientific Python stack.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# scipy.signal is where the peak-finding machinery lives. We use:
#   find_peaks        -- the detector itself (returns indices of local maxima)
#   peak_prominences  -- measures how much each peak stands out (section 6)
#   peak_widths       -- measures each peak's width in samples (section 8)
#   savgol_filter     -- Savitzky-Golay smoothing, from lesson 3.2 (section 11)
from scipy.signal import find_peaks, peak_prominences, peak_widths, savgol_filter

# Sparse tools for the AsLS baseline correction we reuse from lesson 3.3.
from scipy import sparse
from scipy.sparse.linalg import spsolve

# Our UV-Vis simulator. The data carries its own ground truth (the real peak
# positions), so we can check honestly whether detection found them.
from simulated_data import uvvis

# A folder for saved figures (regenerable scratch; git-ignored).
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

# One consistent, readable plot style for the whole notebook.
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# A small, consistent colour vocabulary so every figure reads the same way.
C_SIG   = "#1a73e8"  # the measured signal (blue)
C_TRUE  = "#188038"  # ground truth / correct detections (green)
C_FALSE = "#ea4335"  # false positives / problems (red)
C_AUX   = "#f9ab00"  # helper lines, thresholds (amber)

print("Ready. find_peaks, peak_prominences, peak_widths, and the simulator are loaded.")

## 2. What even is a peak? (and why "local maximum" isn't enough)

Start with the textbook definition: **a peak is a point that is higher than the
points on either side of it** — a local maximum. That's exactly what
`find_peaks` detects at its core.

The problem is that *noise* is full of local maxima. Every little upward jitter
is, technically, higher than its two neighbours. So if you run a bare detector
over a flat, noisy stretch of spectrum — a region with **no real peaks at all** —
you don't get zero peaks. You get dozens.

Let's prove it on a piece of pure baseline noise.

In [ ]:
# A flat region with ONLY noise -- no analyte bands at all.
# We make it by simulating a spectrum with an empty peak list.
flat = uvvis.simulate(peaks=[], noise=0.01, baseline=None, seed=1)
wl_flat, y_flat = flat.x, flat.single()

# The bare detector: "give me every local maximum." No thresholds.
noise_idx, _ = find_peaks(y_flat)

print(f"Real peaks in this region:     0")
print(f"Local maxima find_peaks found: {len(noise_idx)}")

fig, ax = plt.subplots()
ax.plot(wl_flat, y_flat, color=C_SIG, lw=1.0, label="pure noise (no real peaks)")
ax.plot(wl_flat[noise_idx], y_flat[noise_idx], "x", color=C_FALSE, ms=6,
        label=f"{len(noise_idx)} 'peaks' detected")
ax.set_xlabel(f"{flat.x_label} ({flat.x_unit})")
ax.set_ylabel(flat.y_label)
ax.set_title("Every noise wiggle is a local maximum")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "01_what_is_a_peak.png", dpi=150, bbox_inches="tight")
plt.show()

Dozens of "peaks" in a region that contains none. This is the central problem of
peak detection, and it reframes the whole task: the job is **not** finding local
maxima (that part is trivial and useless on its own). The job is **separating the
few maxima that are real bands from the many that are noise.** Every parameter we
meet from here on — height, prominence, distance, width — is a different way of
drawing that line.

## 3. A realistic test spectrum — with known truth

To practise honestly we need a spectrum that behaves like a real one but whose
answer we already know. We'll build **four bands** with deliberately different
characters:

| Band | Centre (nm) | FWHM (nm) | Height (AU) | Character |
|------|-------------|-----------|-------------|-----------|
| A | 450 | 35 | 0.90 | tall, well isolated |
| B | 550 | 24 | 0.55 | medium, close to C |
| C | 600 | 24 | 0.45 | the smallest band, close to B |
| D | 690 | 55 | 0.65 | broad, isolated |

On top of the bands we add realistic **noise** and a **sloping baseline** (a
drifting background that climbs across the spectrum — exactly the kind of thing
lesson 3.3 removes). Because the simulator records the true peak list in
`meta.attrs["peaks"]`, we keep the real centres as our answer key. We also keep a
**noise-free, baseline-free twin** to show what a "perfect" spectrum would look
like.

In [ ]:
# (center_nm, FWHM_nm, amplitude_AU) for each of the four analyte bands.
peaks_true = [
    (450, 35, 0.90),   # A: tall, isolated
    (550, 24, 0.55),   # B: medium, near C
    (600, 24, 0.45),   # C: smallest, near B
    (690, 55, 0.65),   # D: broad, isolated
]

# The messy, realistic spectrum: bands + noise + a sloping background.
ds = uvvis.simulate(
    peaks=peaks_true,
    noise=0.012,
    baseline={"type": "sloping", "slope": 0.0012, "offset": 0.05},
    seed=11,
)
wl, y = ds.x, ds.single()

# The answer key: the true band centres, pulled straight from the ground truth
# the simulator stored for us. THIS is what we grade every detection against.
true_centers = np.array([p[0] for p in ds.meta.attrs["peaks"]], dtype=float)

# A "perfect" twin -- same bands, no noise, no baseline -- for visual reference.
clean = uvvis.simulate(peaks=peaks_true, noise=0.0, baseline=None, seed=11)
y_clean = clean.single()

print("True band centres (our answer key):", true_centers)

fig, ax = plt.subplots()
ax.plot(wl, y, color=C_SIG, lw=1.3, label="measured (noise + sloping baseline)")
ax.plot(wl, y_clean, color="0.5", lw=1.4, ls="--", label="noise-free, flat twin")
# Drop a faint marker at each TRUE centre so we can see where the answers are.
for c in true_centers:
    ax.axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.7)
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title("The test spectrum: four known bands, dressed in noise and drift")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "02_test_spectrum.png", dpi=150, bbox_inches="tight")
plt.show()

Notice two honest complications we built in on purpose. The whole spectrum
**rides up on the sloping baseline** — band D at 690 nm sits on a much higher
background than band A at 450 nm, even though A is the taller *band*. And bands B
and C are **close together** (550 and 600 nm), the kind of near-neighbours that
test whether a detector can keep them apart. Both of these will break the simpler
approaches.

## 4. The naive detector, and why it fails

The temptation is to call `find_peaks(y)` and move on. Let's see what that gives
us on a *real* spectrum (not just the noise region from section 2).

In [ ]:
naive_idx, _ = find_peaks(y)   # every local maximum, no filtering

print(f"True number of bands:     {len(true_centers)}")
print(f"find_peaks(y) detected:   {len(naive_idx)}")

fig, ax = plt.subplots()
ax.plot(wl, y, color=C_SIG, lw=1.2)
ax.plot(wl[naive_idx], y[naive_idx], "x", color=C_FALSE, ms=5,
        label=f"{len(naive_idx)} 'peaks'")
for c in true_centers:
    ax.axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.7)
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title(f"Naive find_peaks: {len(naive_idx)} detections for {len(true_centers)} real bands")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "03_naive_failure.png", dpi=150, bbox_inches="tight")
plt.show()

Around 85 detections for 4 real bands. The detector isn't *wrong* — every red x
really is a local maximum — it's just answering the wrong question. We don't want
"every bump." We want "every bump that is tall enough, distinct enough, and broad
enough to be a real band." That's what the four filtering parameters give us.

## 5. Height — the first, and most fragile, filter

The most obvious filter: **ignore anything below some intensity.** `find_peaks`
calls this `height`. Noise wiggles are small, real bands are tall, so a height
threshold should separate them — and on a flat baseline, it largely does.

The trouble is that **height is measured from zero**, so it is at the mercy of
the baseline. On our sloping spectrum, the background climbs to ~0.5 AU on the
right. So a band's *absolute* height mixes two things: how big the band really is,
and how high the baseline happened to lift it. Watch what that does to any single
threshold.

In [ ]:
# Try a threshold low enough to catch the smallest band (C, ~0.45 AU tall band,
# but sitting on baseline so its top is ~0.75 AU).
HEIGHT = 0.6
idx_h, _ = find_peaks(y, height=HEIGHT)

# Grade it against the truth (we define a proper grader in section 10; here we
# just eyeball the count).
print(f"height >= {HEIGHT}: {len(idx_h)} detections for {len(true_centers)} bands")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)

# Left: a LOW threshold catches the bands but admits a crowd of baseline-lifted
# noise on the right-hand (high-baseline) side.
axes[0].plot(wl, y, color=C_SIG, lw=1.2)
axes[0].axhline(HEIGHT, color=C_AUX, lw=1.3, ls="--", label=f"height = {HEIGHT}")
axes[0].plot(wl[idx_h], y[idx_h], "x", color=C_FALSE, ms=6,
             label=f"{len(idx_h)} detected")
axes[0].set_title("Low threshold: catches bands, admits baseline noise")

# Right: a HIGH threshold rejects the noise but now MISSES the real bands that
# sit on the low part of the baseline.
HIGH = 0.9
idx_hi, _ = find_peaks(y, height=HIGH)
axes[1].plot(wl, y, color=C_SIG, lw=1.2)
axes[1].axhline(HIGH, color=C_AUX, lw=1.3, ls="--", label=f"height = {HIGH}")
axes[1].plot(wl[idx_hi], y[idx_hi], "x", color=C_FALSE, ms=6,
             label=f"{len(idx_hi)} detected")
axes[1].set_title("High threshold: rejects noise, now MISSES real bands")

for ax in axes:
    for c in true_centers:
        ax.axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.6)
    ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
    ax.legend(loc="upper left", fontsize=8)
axes[0].set_ylabel(ds.y_label)
fig.suptitle("Height has no good threshold when the baseline drifts", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(EXPORTS / "04_height.png", dpi=150, bbox_inches="tight")
plt.show()

There is **no single height threshold that works**: low enough to catch every
band lets baseline-lifted noise through; high enough to reject the noise throws
away real bands. Height conflates "tall band" with "tall background." We need a
measure of a peak that ignores where the baseline happens to sit — and that is
prominence.

## 6. Prominence — how much a peak stands out

**Prominence** measures how much a peak rises above the *surrounding terrain*,
not above zero. Picture each peak as a hilltop: its prominence is how far you'd
have to descend into the valleys on either side before you could climb to any
higher peak. A tall band on a high baseline and the same band on a low baseline
have the **same prominence** — because prominence is the height of the peak above
its own local lows, and the baseline cancels out.

That one property is why prominence is the single most useful peak parameter. A
noise wiggle barely rises above its neighbours (tiny prominence); a real band
towers over the local valleys (large prominence). Let's measure it.

In [ ]:
# Find candidate maxima, then ask scipy how prominent each one is.
cand_idx, _ = find_peaks(y)
proms = peak_prominences(y, cand_idx)[0]   # one prominence value per candidate

# The four real bands have large prominence; the noise has tiny prominence.
# Sort to see the gap between "real" and "noise".
order = np.argsort(proms)[::-1]
print("Top prominences (real bands):", np.round(proms[order][:6], 3))
print("Typical noise prominence:    ", np.round(np.median(proms), 4))

# Now detect with a prominence threshold that sits in the gap.
PROM = 0.05
idx_p, _ = find_peaks(y, prominence=PROM)
print(f"\nprominence >= {PROM}: {len(idx_p)} detections "
      f"for {len(true_centers)} bands -> {np.round(wl[idx_p], 0)}")

fig, ax = plt.subplots()
ax.plot(wl, y, color=C_SIG, lw=1.3)
# Draw each detected peak's prominence as a vertical drop from its top down to
# the level it stands above -- a literal picture of "how much it stands out".
det_proms, left_bases, right_bases = peak_prominences(y, idx_p)
for i, pk in enumerate(idx_p):
    base_level = y[pk] - det_proms[i]
    ax.vlines(wl[pk], base_level, y[pk], color=C_FALSE, lw=2.0)
ax.plot(wl[idx_p], y[idx_p], "o", color=C_TRUE, ms=7,
        label=f"{len(idx_p)} peaks (prominence ≥ {PROM})")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title("Prominence = how far each peak rises above its surroundings")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "05_prominence.png", dpi=150, bbox_inches="tight")
plt.show()

A single prominence threshold cleanly recovers **all four bands and nothing
else** — the same task where height could find no working threshold at all. The
red bars show each peak's prominence: the vertical distance from its summit down
to the surrounding valley floor. Because that distance is measured locally, the
sloping baseline simply doesn't enter into it. **If you reach for one parameter
first, reach for prominence.**

## 7. Distance — minimum separation between peaks

`distance` sets the **minimum spacing** (in data points) allowed between
detected peaks; when several candidates fall closer than that, `find_peaks` keeps
only the tallest. It's useful for two jobs: collapsing a *cluster* of noise spikes
that sit right next to each other into nothing, and enforcing physical sense
(two absorbance maxima 1 nm apart are almost never two different bands).

But `distance` cuts both ways. Set it too large and it will **merge two genuinely
separate bands** — exactly the risk for our close neighbours B and C. Let's see
the trade-off directly.

In [ ]:
# Our axis is 400-800 nm over 400 points => ~1 nm per point.
nm_per_point = (wl[-1] - wl[0]) / (len(wl) - 1)
print(f"Axis resolution: {nm_per_point:.2f} nm per point")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)

# A sensible distance: a few points, enough to merge noise clusters but far
# smaller than the ~50 nm gap between real bands B and C.
for ax, dist, title in [
    (axes[0], 8,  "distance = 8 points (~8 nm): keeps B and C apart"),
    (axes[1], 80, "distance = 80 points (~80 nm): merges B and C"),
]:
    idx_d, _ = find_peaks(y, prominence=0.05, distance=dist)
    ax.plot(wl, y, color=C_SIG, lw=1.2)
    ax.plot(wl[idx_d], y[idx_d], "o", color=C_TRUE, ms=7,
            label=f"{len(idx_d)} peaks")
    for c in true_centers:
        ax.axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
    ax.legend(loc="upper right", fontsize=9)
axes[0].set_ylabel(ds.y_label)
fig.suptitle("distance: too small does little, too large merges real bands", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(EXPORTS / "06_distance.png", dpi=150, bbox_inches="tight")
plt.show()

With a small `distance` all four bands survive; with an over-large one, B and C
collapse into a single detection — a **false negative manufactured by a parameter
choice**. The rule of thumb: set `distance` from the *physics* (the narrowest
real spacing you expect to resolve), not by turning the knob until the picture
looks tidy.

## 8. Width — rejecting spikes, selecting bands

A real UV-Vis band has a finite **width** — tens of nanometres of FWHM. A noise
spike, or a single-pixel cosmic-ray hit, is **one or two points wide**. So width
is a powerful discriminator that height and prominence don't capture: it asks
*"is this bump shaped like a band, or like a glitch?"*

`find_peaks` can both *filter* on width and *report* it. Let's add a couple of
sharp artificial spikes to the spectrum — the kind of single-point artefact you
get from electrical noise or a cosmic ray — and use width to reject them while
keeping the real bands.

In [ ]:
# Inject two sharp, 1-point spikes (artefacts) on top of the spectrum.
y_spiky = y.copy()
spike_positions = [500, 640]            # nm
for pos in spike_positions:
    j = int(np.argmin(np.abs(wl - pos)))
    y_spiky[j] += 0.6                   # a tall, razor-thin spike

# Detect with prominence only -- the spikes are prominent, so they slip through.
idx_nw, props_nw = find_peaks(y_spiky, prominence=0.05)
widths_nw = peak_widths(y_spiky, idx_nw, rel_height=0.5)[0]  # FWHM in points
print("Detected (prominence only):", np.round(wl[idx_nw], 0))
print("Their widths (points):     ", np.round(widths_nw, 1))

# Now ADD a width requirement: real bands are many points wide; spikes are ~1-2.
idx_w, _ = find_peaks(y_spiky, prominence=0.05, width=3)
print("\nDetected (prominence + width>=3):", np.round(wl[idx_w], 0))

fig, ax = plt.subplots()
ax.plot(wl, y_spiky, color=C_SIG, lw=1.1)
ax.plot(wl[idx_nw], y_spiky[idx_nw], "x", color=C_FALSE, ms=8,
        label="prominence only (spikes sneak in)")
ax.plot(wl[idx_w], y_spiky[idx_w], "o", color=C_TRUE, ms=7, mfc="none", mew=1.8,
        label="prominence + width ≥ 3 (spikes rejected)")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title("Width tells a band (tens of nm wide) from a spike (1-2 points)")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "07_width.png", dpi=150, bbox_inches="tight")
plt.show()

The spikes are *prominent* — they fool a prominence-only filter — but they are
absurdly *narrow*, so a modest `width` requirement removes them while every real
band stays. Width is the natural defence against single-point artefacts, which
prominence and height alone can't see.

## 9. Putting the four knobs together

Each parameter answers a different question:

- **height** — is it tall enough? (but baseline-dependent)
- **prominence** — does it stand out from its surroundings? (baseline-robust)
- **distance** — is it far enough from the next peak? (resolution / declustering)
- **width** — is it shaped like a band, not a spike?

Used together on the messy spectrum, they turn ~85 raw maxima into exactly the
four real bands.

In [ ]:
idx_final, props = find_peaks(
    y,
    prominence=0.05,   # stands clearly above local noise (the workhorse filter)
    distance=8,        # at least ~8 nm apart (keeps close bands B & C separate)
    width=3,           # at least a few points wide (rejects spikes)
)
detected_centers = wl[idx_final]
print(f"Final detection: {len(idx_final)} peaks at {np.round(detected_centers, 0)}")
print(f"True centres:    {true_centers}")

fig, ax = plt.subplots()
ax.plot(wl, y, color=C_SIG, lw=1.3, label="measured spectrum")
ax.plot(detected_centers, y[idx_final], "o", color=C_TRUE, ms=9, mfc="none",
        mew=2.0, label=f"{len(idx_final)} detected peaks")
for c in true_centers:
    ax.axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.6)
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
ax.set_ylabel(ds.y_label)
ax.set_title("Height + prominence + distance + width → the four real bands")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "08_tuned.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Grading against ground truth: false positives vs. false negatives

"It looks right" is not a result. Because we know the true centres, we can score
any detection exactly. Two kinds of error matter, and they pull in opposite
directions:

- a **false positive** — a detected "peak" that isn't a real band (we invented
  it; usually noise);
- a **false negative** — a real band we **failed** to detect (we missed it).

We match each detected peak to a true centre if it lands within a tolerance
(say, 8 nm). From that we get the familiar pair:

- **precision** = of the peaks we reported, what fraction were real;
- **recall** = of the real bands, what fraction we found.

A loose detector finds everything (recall = 1) but reports junk (low precision).
A strict detector reports only clean peaks (high precision) but starts missing
real bands (recall drops). The art is choosing where to sit on that curve.

In [ ]:
def grade_peaks(detected_wl, true_wl, tol_nm=8.0):
    '''Score detected peak positions against known true centres.

    A detected peak counts as a true positive if it lands within `tol_nm` of a
    true centre (each true centre can be matched only once). Returns a small dict
    of counts and the precision/recall summary.
    '''
    detected_wl = np.atleast_1d(detected_wl)
    matched_truth = set()
    true_positives = 0
    for d in detected_wl:
        # nearest unmatched true centre
        diffs = np.abs(true_wl - d)
        j = int(np.argmin(diffs)) if len(diffs) else -1
        if j >= 0 and diffs[j] <= tol_nm and j not in matched_truth:
            matched_truth.add(j)
            true_positives += 1
    n_detected = len(detected_wl)
    false_positives = n_detected - true_positives          # invented peaks
    false_negatives = len(true_wl) - true_positives        # missed real bands
    precision = true_positives / n_detected if n_detected else 0.0
    recall = true_positives / len(true_wl) if len(true_wl) else 0.0
    return {
        "detected": n_detected,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
    }

# Grade the naive detector vs. the tuned one.
print("Naive find_peaks(y):     ", grade_peaks(wl[naive_idx], true_centers))
print("Tuned (prom+dist+width): ", grade_peaks(detected_centers, true_centers))

The naive detector has perfect recall (it finds all four bands — it finds
*everything*) but disastrous precision. The tuned detector reaches precision =
recall = 1.0: four real bands, zero invented. Now let's watch the trade-off move
as we sweep the prominence threshold from too-loose to too-strict.

In [ ]:
prom_grid = np.linspace(0.0, 0.6, 25)
fp_count, fn_count = [], []
for pr in prom_grid:
    idx_pr, _ = find_peaks(y, prominence=pr) if pr > 0 else find_peaks(y)
    g = grade_peaks(wl[idx_pr], true_centers)
    fp_count.append(g["false_positives"])
    fn_count.append(g["false_negatives"])

fig, ax = plt.subplots()
ax.plot(prom_grid, fp_count, "-o", color=C_FALSE, ms=4,
        label="false positives (invented peaks)")
ax.plot(prom_grid, fn_count, "-s", color=C_SIG, ms=4,
        label="false negatives (missed bands)")
ax.axvspan(0.05, 0.45, color=C_TRUE, alpha=0.12, label="usable window (0 errors)")
ax.set_yscale("symlog")            # FP spans 0..~80; symlog keeps both visible
ax.set_xlabel("prominence threshold")
ax.set_ylabel("error count (symlog)")
ax.set_title("The false-positive ↔ false-negative trade-off")
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "09_grading.png", dpi=150, bbox_inches="tight")
plt.show()

print("Too loose (prom≈0):  many false positives, zero misses.")
print("Too strict (prom>0.45): zero false positives, but real bands start to vanish.")
print("The flat valley between them is the window you want to sit in.")

This curve is the whole game in one picture. At low prominence you drown in false
positives; push too high and the small bands (B and C, the least prominent)
disappear as false negatives. There is a **comfortable window** in between where
both errors are zero — but on noisier or messier data that window narrows, and on
real data you can't draw this plot because you don't have the answer key. Which is
exactly why the next section matters: **the best way to widen the window is to
clean the spectrum first.**

## 11. Preprocessing changes everything: smoothing and baseline correction

Here is the lesson's biggest practical point: **most peak-detection failures are
really preprocessing failures.** The two previous notebooks give us the fixes.

- **Smoothing (3.2)** suppresses the noise wiggles that masquerade as peaks, so a
  *looser, safer* threshold stops admitting false positives.
- **Baseline correction (3.3)** removes the drift that made `height` meaningless,
  so simple thresholds become transferable across the whole spectrum.

First, smoothing. We deliberately use a **loose** prominence threshold (0.02) —
one that drowned in false positives earlier — and watch smoothing rescue it.

In [ ]:
# Savitzky-Golay smoothing from lesson 3.2: a gentle window, low polynomial order.
y_smooth = savgol_filter(y, window_length=15, polyorder=3)

LOOSE = 0.02
raw_idx, _ = find_peaks(y, prominence=LOOSE)
sm_idx, _ = find_peaks(y_smooth, prominence=LOOSE)
print(f"Loose prominence ({LOOSE}) on RAW:      ", grade_peaks(wl[raw_idx], true_centers))
print(f"Loose prominence ({LOOSE}) on SMOOTHED: ", grade_peaks(wl[sm_idx], true_centers))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
for ax, sig, idx, title in [
    (axes[0], y, raw_idx, f"RAW: {len(raw_idx)} detections (noise false positives)"),
    (axes[1], y_smooth, sm_idx, f"SMOOTHED: {len(sm_idx)} detections (clean)"),
]:
    ax.plot(wl, sig, color=C_SIG, lw=1.2)
    ax.plot(wl[idx], sig[idx], "x", color=C_FALSE, ms=6)
    for c in true_centers:
        ax.axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.6)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})")
axes[0].set_ylabel(ds.y_label)
fig.suptitle("Same loose threshold: smoothing removes the noise false positives", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(EXPORTS / "10_preprocessing.png", dpi=150, bbox_inches="tight")
plt.show()

Same detector, same threshold — the only change is that we cleaned the noise
first, and a flood of false positives became a clean four. Now baseline
correction, which is what makes a plain **height** threshold usable again. We
reuse the AsLS routine from lesson 3.3.

In [ ]:
def asls_baseline(y, lam=1e5, p=0.01, n_iter=10):
    '''Asymmetric Least Squares baseline (Eilers & Boelens), from lesson 3.3.'''
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    w = np.ones(L)
    for _ in range(n_iter):
        W = sparse.spdiags(w, 0, L, L)
        z = spsolve(W + lam * (D @ D.T), w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z

y_corr = y - asls_baseline(y)     # baseline-corrected spectrum, sits near zero

# A single HEIGHT threshold now works everywhere, because the baseline is flat.
HEIGHT = 0.35
raw_h, _ = find_peaks(y,      height=HEIGHT, distance=8, width=3)
cor_h, _ = find_peaks(y_corr, height=HEIGHT, distance=8, width=3)
print(f"height>={HEIGHT} on RAW (sloping):   ", grade_peaks(wl[raw_h], true_centers))
print(f"height>={HEIGHT} on BASELINE-CORR.:  ", grade_peaks(wl[cor_h], true_centers))

On the sloping spectrum, a single height threshold lets baseline-lifted noise
through on the high-background side (false positives); after AsLS the background
is flat, the *same* threshold means the same thing at every wavelength, and the
four bands come out cleanly. **The reliable recipe is: baseline-correct →
(optionally) smooth → detect.** Detection is the last and easiest step; the
quality was decided upstream.

## 12. The overlapping-peak limit

There is one failure that *no* choice of detection parameters can fix: when two
bands overlap so heavily that together they form a **single maximum**. There is
only one summit, so a peak finder — which by definition looks for maxima — can
report only one peak. The second band is not noise and not missed by a bad
threshold; it is genuinely *invisible to maxima-finding*.

In [ ]:
# Two heavily-overlapping bands, ~30 nm apart with ~40 nm FWHM each.
# We use a NOISE-FREE spectrum here on purpose: the point is that overlap is an
# *intrinsic* limit of maxima-finding, not something noise caused. If the clean
# case already fails, no amount of denoising will save it.
overlap_true = [(545, 40, 0.60), (575, 40, 0.50)]
ov = uvvis.simulate(peaks=overlap_true, noise=0.0, baseline=None, seed=3)
wl_o, y_o = ov.x, ov.single()
ov_centers = np.array([p[0] for p in ov.meta.attrs["peaks"]])

# Maxima-finding sees only ONE peak, even on this perfectly clean spectrum.
idx_o, _ = find_peaks(y_o, prominence=0.02)
print(f"True bands: {ov_centers}")
print(f"find_peaks detected: {np.round(wl_o[idx_o], 0)}  ({len(idx_o)} peak for 2 bands)")

# The classic hint: the SECOND DERIVATIVE has a minimum under each component,
# so two negative lobes reveal the hidden pair even when the raw curve shows one.
d2 = savgol_filter(y_o, window_length=31, polyorder=3, deriv=2)
d2_min_idx, _ = find_peaks(-d2, prominence=1e-5)   # minima of d2 = maxima of -d2
print(f"2nd-derivative minima reveal: {np.round(wl_o[d2_min_idx], 0)}  (both bands!)")

fig, axes = plt.subplots(2, 1, figsize=(9, 6.5), sharex=True)
axes[0].plot(wl_o, y_o, color=C_SIG, lw=1.5)
axes[0].plot(wl_o[idx_o], y_o[idx_o], "o", color=C_FALSE, ms=8,
             label="find_peaks: 1 peak")
for c in ov_centers:
    axes[0].axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.7)
axes[0].set_ylabel(ov.y_label)
axes[0].set_title("Two bands, one summit: detection sees a single peak")
axes[0].legend(loc="upper right", fontsize=9)

axes[1].plot(wl_o, d2, color="0.4", lw=1.3)
axes[1].plot(wl_o[d2_min_idx], d2[d2_min_idx], "v", color=C_TRUE, ms=9,
             label="2nd-derivative minima: 2 components")
for c in ov_centers:
    axes[1].axvline(c, color=C_TRUE, lw=1.0, ls=":", alpha=0.7)
axes[1].set_xlabel(f"{ov.x_label} ({ov.x_unit})")
axes[1].set_ylabel("2nd derivative")
axes[1].set_title("The second derivative resolves what the raw curve hides")
axes[1].legend(loc="upper right", fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "11_overlap.png", dpi=150, bbox_inches="tight")
plt.show()

The raw spectrum has one summit; detection honestly reports one peak. The
**second derivative** is sharper than the original (each component makes its own
negative lobe), so it can *flag* that two bands are present — a standard trick for
spotting shoulders. But flagging is not quantifying: to get the two centres,
heights, and areas you need to **fit** a model of two overlapping bands to the
data. That is the subject of lesson 3.6 (peak fitting). The honest takeaway is
knowing the boundary: **detection locates resolved peaks; fitting is required to
separate overlapping ones.**

## 13. A reusable `detect_peaks()` helper

Finally, wrap the workflow into one small function with sensible defaults and a
tidy return — a tuple of peak positions plus their measured properties — so later
notebooks can detect peaks in one call without re-deriving any of this.

In [ ]:
def detect_peaks(x, y, prominence=0.05, distance=8, width=3, return_properties=False):
    '''Detect peaks and report them in axis units.

    A thin, opinionated wrapper around scipy.signal.find_peaks with defaults
    chosen for cleaned UV-Vis spectra (baseline-corrected, lightly smoothed).
    Tune `prominence` first; it is the most robust knob.

    Parameters
    ----------
    x, y : array
        The axis (e.g. wavelength) and the (ideally cleaned) intensity.
    prominence : float
        Minimum prominence -- how far a peak must rise above its surroundings.
    distance : int
        Minimum spacing between peaks, in data points.
    width : float
        Minimum peak width, in data points (rejects single-point spikes).
    return_properties : bool
        If True, also return a dict with each peak's height, prominence, and
        FWHM (in axis units).

    Returns
    -------
    positions : ndarray
        Peak positions in the units of `x`.
    properties : dict, optional
        Present only if return_properties=True.
    '''
    idx, props = find_peaks(y, prominence=prominence, distance=distance, width=width)
    positions = x[idx]
    if not return_properties:
        return positions

    # Convert width from points to axis units so it's physically meaningful.
    step = (x[-1] - x[0]) / (len(x) - 1)
    properties = {
        "indices": idx,
        "height": y[idx],
        "prominence": props["prominences"],
        "fwhm": props["widths"] * step,        # points -> nm (or whatever x is)
    }
    return positions, properties

# Demonstrate on a fully cleaned version of our test spectrum.
y_ready = savgol_filter(y - asls_baseline(y), 15, 3)   # baseline-correct, then smooth
positions, properties = detect_peaks(wl, y_ready, return_properties=True)

print("Detected band centres (nm):", np.round(positions, 1))
print("Heights (AU):              ", np.round(properties["height"], 3))
print("Prominences (AU):          ", np.round(properties["prominence"], 3))
print("FWHM (nm):                 ", np.round(properties["fwhm"], 1))
print("\nGraded against truth:      ", grade_peaks(positions, true_centers))

## Key Takeaways

- **A peak is a local maximum, but most local maxima are noise.** Detection is
  not "find the maxima" (trivial and useless alone) — it's *separating real bands
  from noise*. A bare `find_peaks` returned ~85 maxima for 4 real bands.
- **Height is fragile because it's measured from zero.** When the baseline drifts,
  no single height threshold both catches every band and rejects the noise.
- **Prominence is the workhorse.** It measures how far a peak rises above its
  *local* surroundings, so it ignores baseline level. One prominence threshold
  recovered all four bands and nothing else.
- **Distance and width encode physics.** `distance` is your expected resolution
  (too large merges real neighbours); `width` rejects single-point spikes that
  prominence can't see.
- **Grade, don't admire.** With known truth, score detections as true positives,
  false positives (invented), and false negatives (missed). The
  prominence sweep makes the false-positive ↔ false-negative trade-off explicit.
- **Detection quality is mostly a preprocessing story.** Smoothing (3.2) kills
  noise false positives; baseline correction (3.3) makes thresholds transferable.
  The reliable order is **baseline-correct → smooth → detect.**
- **Overlapping peaks are a hard limit.** Two bands under one summit give one
  maximum; the second derivative can *flag* the pair, but separating them needs
  *fitting* (lesson 3.6), not detection.

## Practical Checklist

- [ ] **Clean first.** Baseline-correct, then lightly smooth, *before* detecting.
- [ ] **Tune prominence first** — it's the most robust knob. Start near a value
  that sits in the gap between band and noise prominences.
- [ ] **Set `distance` from the narrowest spacing you expect to resolve**, not by
  eye.
- [ ] **Set `width` to reject spikes** (a few points) while passing real bands.
- [ ] **Check both error types**: any invented peaks (false positives)? any missed
  bands (false negatives)?
- [ ] **Report peak properties in axis units** (nm), not array indices.
- [ ] **If a "peak" looks like a shoulder, check the second derivative** before
  trusting a single-peak result.

## Common Mistakes

- **Calling `find_peaks(y)` with no parameters** and trusting the count. You'll
  detect noise by the dozen.
- **Filtering on `height` over a drifting baseline.** You're thresholding the
  background as much as the band. Correct the baseline first, or use prominence.
- **Setting `distance` too large** to "clean up" the plot — it silently merges
  genuine neighbouring bands into one.
- **Forgetting `width`**, so a single-pixel spike or cosmic ray is reported as a
  band.
- **Tuning parameters until the picture looks nice** instead of grading against
  something. On real data, sanity-check on a sample with known peaks.
- **Expecting detection to resolve overlapping bands.** It can't; that's a fitting
  problem.

## Reporting Guidance

State the detector and *all* its parameters, plus the preprocessing that preceded
it: **"Peaks detected with `scipy.signal.find_peaks` (prominence = …, distance =
… points, width = … points) on spectra that were AsLS baseline-corrected (λ = …,
p = …) and Savitzky–Golay smoothed (window = …, order = …)."** Peak detection
parameters change which peaks exist in your results, so they are part of the
method and must be reproducible. When you can, report the tolerance and the
reference used to validate detection (e.g. agreement with known band positions).

## Next Lesson

**3.5 — Peak Integration and Quantifying Area.** Now that we can *locate* peaks
reliably, the next question is *how much* — turning a detected band into a number
(its area) that tracks concentration. We'll define integration limits honestly,
handle the residual baseline under a peak, and see why area is often a steadier
measurement than height.